# RAG Academic Paper QA System — End-to-End Demo

This notebook walks through the full pipeline:
1. **Setup** — install dependencies, set API key
2. **Ingest** — parse PDFs and build `chunks.json`
3. **Build Index** — construct BM25 + FAISS indexes
4. **Single Query Demo** — retrieve → rerank → generate
5. **Retrieval Comparison** — BM25 vs Dense vs Hybrid
6. **Evaluation Results** — RAGAS + Recall/MRR/NDCG

> **Quick start**: If you already have `indexes/chunks.json` and the index files,
> skip to Section 4.

## 1. Setup

In [ ]:
# Install dependencies (run once)
# !pip install -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

# Add src/ to path so we can import project modules
sys.path.insert(0, str(Path("../src").resolve()))

import config

print("Project root:", config.ROOT_DIR)
print("Papers dir  :", config.PAPERS_DIR, "| exists:", config.PAPERS_DIR.exists())
print("Chunks path :", config.CHUNKS_PATH, "| exists:", config.CHUNKS_PATH.exists())
print("Groq API key:", "[SET]" if config.GROQ_API_KEY else "[NOT SET — placeholder mode]")

## 2. Ingest — Parse PDFs into Chunks

Place PDF files in `data/papers/`, or use the `--arxiv` flag to download from ArXiv.

In [ ]:
from ingest import ingest_all, download_arxiv_papers

# Option A: Process PDFs already in data/papers/
# chunks = ingest_all()

# Option B: Download from ArXiv first (requires internet)
# download_arxiv_papers(max_results=10, query="retrieval augmented generation")
# chunks = ingest_all()

# For now, just check what's there
pdf_files = sorted(config.PAPERS_DIR.glob("*.pdf"))
print(f"PDFs in data/papers/: {len(pdf_files)}")
for p in pdf_files[:5]:
    print(f"  {p.name}")

## 3. Build Index — BM25 + FAISS

Run once after ingestion. Skip if `indexes/` already has the artifacts.

In [ ]:
from build_index import build_all

# build_all()          # skip if already built
# build_all(rebuild=True)  # force rebuild

print("BM25 index  :", "exists" if config.BM25_INDEX_PATH.exists() else "NOT BUILT")
print("FAISS index :", "exists" if config.FAISS_INDEX_PATH.exists() else "NOT BUILT")

## 4. Single Query Demo

In [ ]:
from pipeline import RAGPipeline

# Initialize the full pipeline (loads all components)
# Requires: chunks.json + bm25.pkl + faiss_index/ to exist
try:
    pipeline = RAGPipeline(retrieval_mode="hybrid", top_k=20, rerank_top_k=5)
    PIPELINE_READY = True
except FileNotFoundError as e:
    print(f"Pipeline not ready: {e}")
    PIPELINE_READY = False

In [ ]:
if PIPELINE_READY:
    QUESTION = "What is the key contribution of the Transformer model?"

    result = pipeline.query(QUESTION)

    print(f"Question : {result['question']}")
    print(f"\nAnswer   :\n{result['answer']}")
    print(f"\nSources  :")
    for i, src in enumerate(result["sources"], 1):
        print(f"  {i}. {src.get('source_paper', 'Unknown')}, p.{src.get('page_num', '?')} "
              f"[score={src.get('rerank_score', src.get('score', 0)):.3f}]")
    print(f"\nLatency  : {result['latency']}")

## 5. Retrieval Comparison — BM25 vs Dense vs Hybrid

For a single query, compare the top-5 results from each retrieval mode.

In [ ]:
if PIPELINE_READY:
    from retrieval import load_retriever

    retriever = pipeline.retriever
    query = "How does BERT use masked language modeling?"

    for mode in ["bm25", "dense", "hybrid"]:
        print(f"\n{'─'*50}")
        print(f"Mode: {mode.upper()} — Top 3 results")
        print(f"{'─'*50}")
        results = retriever.retrieve(query, mode=mode, top_k=3)
        for i, r in enumerate(results, 1):
            print(f"  {i}. [{r['chunk_id']}] score={r['score']:.4f}")
            print(f"     {r['chunk_text'][:120].strip()}...")

## 6. Evaluation Results

Requires `data/qa_testset.json` with annotated `relevant_chunk_ids`.

In [ ]:
from evaluate import load_testset, run_retrieval_eval, run_reranking_ablation

testset = load_testset()
print(f"Test questions with annotations: {len(testset)}")

In [ ]:
# Experiment 1: Retrieval comparison
# (Uncomment when testset is annotated and indexes are built)

# results = run_retrieval_eval(testset=testset)
# import pandas as pd
# df = pd.DataFrame(results).T
# display(df)

In [ ]:
# Experiment 2: Reranking ablation

# ablation = run_reranking_ablation(testset=testset)
# df_abl = pd.DataFrame(ablation).T
# display(df_abl)

In [ ]:
# Experiment 3: RAGAS end-to-end (requires Groq API key)

# from evaluate import run_ragas_eval
# ragas_scores = run_ragas_eval(testset=testset)
# print(ragas_scores)